# ⚖️ Lekcja 7: Uczciwy sędzia — dlaczego jeden `train_test_split` może kłamać i jak działa K-Fold Cross-Validation?

W Lekcjach 4 i 6 ocenialiśmy nasz model zawsze w ten sam sposób: za pomocą `train_test_split(..., test_size=0.2, random_state=42)` odkładaliśmy do sejfu **292 domy (20%)**, a na pozostałych **1168 domach (80%)** uczyliśmy model.

Ale zatrzymajmy się na chwilę i zadajmy sobie pytanie prawdziwego naukowca:
> 🤔 **„A co, jeśli akurat te 292 domy, które wylosowaliśmy do sejfu przy `random_state=42`, były wyjątkowo łatwe (albo wyjątkowo trudne) do wyceny?”**

---

### 1. 🎲 Loteria jednego sprawdzianu (Dlaczego pojedynczy `train_test_split` bywa złudny?)

Wyobraź sobie ucznia, który przygotowuje się do egzaminu z historii z podręcznika mającego **5 dużych rozdziałów** (Starożytność, Średniowiecze, Nowożytność, XIX wiek, XX wiek).
Nauczyciel zamiast przepytać go z całej książki, rzuca kostką i mówi: *„Dzisiaj sprawdzian tylko z Rozdziału nr 1!”*.
* Jeśli uczeń akurat świetnie zna Starożytność $\rightarrow$ dostaje **5+** i myśli, że umie całą historię.
* Jeśli akurat w Rozdziale nr 3 były najtrudniejsze daty, których nie powtórzył $\rightarrow$ nawet się o tym nie dowie, bo ten rozdział nie trafił na sprawdzian!

Dokładnie tak samo działa pojedynczy `train_test_split`! W zależności od tego, jaką liczbę wpiszesz w `random_state` (jak potasujesz talię 1460 domów), do sejfu trafiają inne 292 domy, a średni błąd tego samego modelu potrafi skakać o **kilka tysięcy dolarów**!

---

### 2. 🔄 Rozwiązanie: Walidacja Krzyżowa (*K-Fold Cross-Validation*) — 5 egzaminów po kolei!

Aby całkowicie wyeliminować ślepy traf, profesjonaliści używają metody **K-Fold Cross-Validation** (najczęściej dla **`K = 5`**, stąd nazwa *5-Fold* — „5 części / zakładek”).

Jak to działa krok po kroku?
1. Dzielimy całą tabelę **1460 domów** na **5 równych kawałków tortu (5 Foldów)** — każdy kawałek ma dokładnie **292 domy (20%)**.
2. Przeprowadzamy **5 niezależnych egzaminów** (w każdym egzaminie model startuje od zera, z czystą pamięcią!):

| Numer egzaminu (Runda) | Kawałek #1 (292 domy) | Kawałek #2 (292 domy) | Kawałek #3 (292 domy) | Kawałek #4 (292 domy) | Kawałek #5 (292 domy) | Wynik rundy |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Fold 1 (Egzamin 1)** | 🔒 **EGZAMIN** | 📚 Nauka | 📚 Nauka | 📚 Nauka | 📚 Nauka | Błąd $\text{MAE}_1$ |
| **Fold 2 (Egzamin 2)** | 📚 Nauka | 🔒 **EGZAMIN** | 📚 Nauka | 📚 Nauka | 📚 Nauka | Błąd $\text{MAE}_2$ |
| **Fold 3 (Egzamin 3)** | 📚 Nauka | 📚 Nauka | 🔒 **EGZAMIN** | 📚 Nauka | 📚 Nauka | Błąd $\text{MAE}_3$ |
| **Fold 4 (Egzamin 4)** | 📚 Nauka | 📚 Nauka | 📚 Nauka | 🔒 **EGZAMIN** | 📚 Nauka | Błąd $\text{MAE}_4$ |
| **Fold 5 (Egzamin 5)** | 📚 Nauka | 📚 Nauka | 📚 Nauka | 📚 Nauka | 🔒 **EGZAMIN** | Błąd $\text{MAE}_5$ |

3. Na samym końcu liczymy **średnią arytmetyczną ze wszystkich 5 egzaminów**:
   $$\text{Wynik K-Fold} = \frac{\text{MAE}_1 + \text{MAE}_2 + \text{MAE}_3 + \text{MAE}_4 + \text{MAE}_5}{5}$$

💡 **Dlaczego to jest tak genialne?** Spójrz na kolumny z kłódką 🔒 w tabeli wyżej: **każdy z 1460 domów trafił do sejfu egzaminacyjnego dokładnie JEDEN raz!** Żadna trudna willa ani dziwna anomalia nie ukryje się już w zbiorze treningowym!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Wczytujemy dane z folderu ../data/ i przygotowujemy nasze 12 cech z Lekcji 6 (8 liczbowych + 4 porządkowe)
train = pd.read_csv('../data/train.csv')

mapa_jakosci = {'Brak': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
for kol in ['KitchenQual', 'ExterQual', 'BsmtQual', 'FireplaceQu']:
    train[kol + '_Num'] = train[kol].fillna('Brak').map(mapa_jakosci)

cechy = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF',
    'YearBuilt', 'YearRemodAdd', 'LotArea', 'BsmtFinSF1',
    'KitchenQual_Num', 'ExterQual_Num', 'BsmtQual_Num', 'FireplaceQu_Num'
]

X = train[cechy]
y = train['SalePrice']

# --- CZĘŚĆ A: Dowód na to, jak pojedynczy train_test_split zależy od szczęścia w losowaniu ---
losowania = [99, 1, 123, 7, 42]
bledy_split = []

print('--- 1. Ten sam model oceniany pojedynczym train_test_split przy różnych tasowaniach kart ---')
for rs in losowania:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=rs)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    blad = mean_absolute_error(y_val, pred)
    bledy_split.append(blad)
    print(f'   Tasowanie random_state={rs:3d} -> Błąd MAE w sejfie: {blad:,.0f} $')

# --- CZĘŚĆ B: Uczciwy sędzia — 5-krotna Walidacja Krzyżowa (5-Fold Cross-Validation) ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_kfold = RandomForestRegressor(n_estimators=100, random_state=42)

# Funkcja cross_val_score automatycznie robi za nas 5 treningów (.fit) i 5 egzaminów (.predict)!
wyniki_5_foldow = -cross_val_score(model_kfold, X, y, cv=kf, scoring='neg_mean_absolute_error')
srednia_kfold = wyniki_5_foldow.mean()

print('\n--- 2. Ten sam model oceniany przez 5-Fold Cross-Validation (każdy dom przetestowany!) ---')
for numer, wynik in enumerate(wyniki_5_foldow, 1):
    print(f'   Fold {numer} (Egzamin {numer}): {wynik:,.0f} $')
print(f'👉 UCZCIWA ŚREDNIA Z 5 FOLDÓW: {srednia_kfold:,.0f} $')

# --- WIZUALIZACJA NA DWÓCH WYKRESACH OBOK SIEBIE ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Wykres lewy: Loteria pojedynczego train_test_split
etykiety_rs = [f'Losowanie\n(state={rs})' for rs in losowania]
ax1.bar(etykiety_rs, bledy_split, color='skyblue', edgecolor='black', width=0.55)
ax1.set_title('1. Loteria jednego podziału (train_test_split)')
ax1.set_ylabel('Średni błąd wyceny MAE ($)')
ax1.set_ylim(0, 25000)
for i, v in enumerate(bledy_split):
    ax1.text(i, v + 500, f'{v:,.0f} $', ha='center', fontweight='bold')
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: 5 Foldów w K-Fold Cross-Validation
etykiety_foldow = [f'Fold {i}\n(292 domy)' for i in range(1, 6)]
kolory_foldow = ['lightgreen', 'lightgreen', 'salmon', 'lightgreen', 'lightgreen']
ax2.bar(etykiety_foldow, wyniki_5_foldow, color=kolory_foldow, edgecolor='black', width=0.55)
ax2.axhline(srednia_kfold, color='darkred', linestyle='--', linewidth=2.5, label=f'Średnia z 5 foldów: {srednia_kfold:,.0f} $')
ax2.set_title('2. Wyniki w 5-Fold Cross-Validation (Zwróć uwagę na Fold 3!)')
ax2.set_ylabel('Średni błąd wyceny MAE ($)')
ax2.set_ylim(0, 25000)
for i, v in enumerate(wyniki_5_foldow):
    ax2.text(i, v + 500, f'{v:,.0f} $', ha='center', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 🔬 Szczegółowa rozbiórka wykresów, zagadki Foldu nr 3 i składni `cross_val_score`

#### 1. Spójrz na lewy wykres: Jak jeden podział mógłby nas oszukać?
Gdybyśmy w Lekcji 4 zamiast `random_state=42` wpisali przypadkiem `random_state=99`, zobaczylibyśmy błąd **`14 714 $`** (prawie **3 000 \$ mniej**!), chociaż nasz model nie stał się ani odrobinę mądrzejszy — po prostu przy tasowaniu nr 99 do sejfu wpadły same grzeczne, typowe domki z przedmieść!

---

#### 2. 🕵️‍♂️ Wielka zagadka prawego wykresu: Dlaczego czerwony słupek `Fold 3` wystrzelił aż do `22 766 $`?!
Spójrz na prawy wykres: w Foldach 1, 2, 4 i 5 błąd naszego modelu wynosi spokojne **16 200 – 17 970 \$**.  
Nagle w **Foldzie 3** błąd podskakuje aż do **`22 766 $`**! Co takiego ukrywało się w trzecim kawałku tortu?

Odpowiedź znasz już z **Lekcji 2**!  
Pamiętasz wykres punktowy metrażu `GrLivArea` i **dwie czerwone kropki w prawym dolnym rogu** (dwa gigantyczne pałace o powierzchni `4676` i `5642` stóp kwadratowych, które sprzedano w stanie nieukończonym za śmieszne `184 750 $` i `160 000 $`)?
* Kiedy robiliśmy pojedynczy `train_test_split(random_state=42)` (który bada tylko kawałek nr 1), **obie te anomalie siedziały ukryte w zbiorze treningowym (`80%`)**, więc nie psuły nam wyniku na egzaminie!
* Dopiero gdy zaprzęgliśmy do pracy **5-Fold Cross-Validation**, w trzeciej rundzie (**Fold 3**) oba te dziwne pałace trafiły na arkusz egzaminacyjny! Model zobaczył pałac o powierzchni 5642 stóp, wycenił go uczciwie na ponad 600 000 \$, a w kluczu odpowiedzi było 160 000 \$ (pomyłka o prawie pół miliona dolarów na jednym domu!).
* **Wniosek:** Dzięki K-Fold Cross-Validation odkryliśmy słaby punkt naszych danych, którego pojedynczy `train_test_split` w ogóle nam nie pokazał!

---

#### 3. Nowe elementy kodu Pythona i zagadka „podwójnego minusa” w `-cross_val_score(..., scoring='neg_mean_absolute_error')`

* **`kf = KFold(n_splits=5, shuffle=True, random_state=42)`** — przygotowujemy krajalnicę do tortu:
  * `n_splits=5` (*number of splits* — liczba podziałów): pokrój dane na **5 równych części**.
  * `shuffle=True` (*tasuj = Prawda*): zanim pokroisz tabelę na 5 części, **potasuj wiersze**.
* **`cross_val_score(model_kfold, X, y, cv=kf, ...)`** — to automat egzaminacyjny. W jednej linijce kodu sam robi pętlę `for` przez 5 foldów, 5 razy wywołuje `.fit()`, 5 razy `.predict()` i zwraca tablicę 5 wyników.
* **Dlaczego wpisujemy `'neg_mean_absolute_error'` (z przedrostkiem `neg_`) i stawiamy znak minus `-` na początku linijki?**  
  To słynna dziwaczność biblioteki `scikit-learn`, o którą pyta każdy początkujący:
  1. Twórcy `sklearn` ustalili żelazną zasadę dla wszystkich narzędzi w bibliotece: **„Im WIĘKSZY wynik, tym LEPSZY model”** (jak procenty na egzaminie: 90% jest lepsze niż 70%).
  2. Ale błąd wyceny (`MAE`) działa odwrotnie: **im MNIEJSZY błąd (np. 16 000 \$ zamiast 22 000 \$), tym lepiej!**
  3. Żeby dopasować błąd do zasady *„więcej = lepiej”*, `sklearn` mnoży błąd przez `-1` (stąd nazwa **`neg_`** od *negative* — ujemny) i zwraca liczby ujemne: `[-17959, -16854, -22766, -17975, -16215]` (bo na osi liczbowej `-16 215` jest większe niż `-22 766`!).
  4. Ponieważ jednak żaden człowiek nie mówi *„pomyliłem się o minus 18 tysięcy dolarów”*, na samym początku stawiamy nasz własny minus: **`-cross_val_score(...)`**. Ze szkolnej matematyki wiemy, że **minus przed minusem daje plus**, więc z powrotem dostajemy czytelne, dodatnie dolary!

In [ ]:
# 🎯 ĆWICZENIE: Co się stanie z wynikiem 5-Fold CV, gdy wyrzucimy 2 podejrzane anomalie z Lekcji 2?
#
# Przed chwilą odkryliśmy, że Fold 3 miał błąd aż 22 766 $, bo wpadły do niego dwa gigantyczne domy
# (GrLivArea > 4000) sprzedane poniżej 300 000 $. Usuńmy te 2 wiersze z tabeli `train` i sprawdźmy wynik 5-Fold!

train_bez_anomalii = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
X_czyste = train_bez_anomalii[cechy]
y_czyste = train_bez_anomalii['SalePrice']

# 1. Oblicz nowe wyniki 5-Fold Cross-Validation dla `X_czyste` oraz `y_czyste`, używając:
#    nowe_wyniki = -cross_val_score(model_kfold, X_czyste, y_czyste, cv=kf, scoring='neg_mean_absolute_error')
# 2. Wypisz tablicę `nowe_wyniki` oraz nową średnią `nowe_wyniki.mean()` za pomocą print()!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Pułapka pojedynczego podziału (`train_test_split`):** Sam wybór ziarna losowości (`random_state=99` vs `42`) potrafi zmienić wynik dokładnie tego samego modelu o niemal 3 000 \$, w zależności od tego, jak trudne domy przypadkiem trafią do sejfu.
2. **Walidacja krzyżowa (`KFold(n_splits=5, shuffle=True, random_state=42)`):** Dzieląc zbiór treningowy na 5 równych części i przeprowadzając 5 rund egzaminacyjnych (`cross_val_score`), gwarantujemy, że każdy z 1460 domów zostanie przetestowany w sejfie dokładnie jeden raz.
3. **Wykrywanie wpływu anomalii (*Outlierów*):** Walidacja 5-Fold natychmiast ujawniła w Foldzie 3 (`22 766 $`) niszczący wpływ dwóch anomalii z Lekcji 2 (`GrLivArea > 4000`) — po ich usunięciu średni błąd z 5 foldów spada z `18 354 $` do `17 324 $`.